# 21 · 手撸 BM25 + dense + RRF 融合

> **学习目标**：80 行手撸 BM25（不依赖 rank_bm25 / Elasticsearch），与 dense embedding 检索做 RRF 融合，看 hybrid 在「关键词命中 vs 语义命中」各自擅长的 query 上如何兜底。
>
> **预备**：16、19、20 跑过。
>
> **为什么重要**：纯 dense 检索对**精确关键词**（型号、人名、ID）经常召不准——文本里写了 "XZ4054H" 你问 "XZ4054H" embedding 都未必匹配最准。BM25 在这种场景一招制敌。**hybrid 是 RAG 生产标配**。

In [ ]:
MODE = 'OFFLINE'        # 'ONLINE' 用 Ollama nomic-embed-text

import numpy as np, hashlib, re, math, requests
from collections import Counter
from typing import Callable
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text: str, dim: int = 256) -> np.ndarray:
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text: str) -> np.ndarray:
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); embed = ollama_embed; print('✅ ONLINE')
    except Exception: MODE = 'OFFLINE'; embed = fake_embed; print('⚠ Ollama 未启动，降级 OFFLINE')
else:
    embed = fake_embed; print('使用 OFFLINE fake_embed')

## 1. 中英混合的简易 tokenizer

BM25 需要把文本切成 token 列表。中文每字一 token，英文按非字母数字切。**避开 jieba 等大依赖**，对 BM25 信号已经够用。

In [ ]:
def tokenize(text: str) -> list[str]:
    """
    - 连续 ASCII 字母 + 数字 = 一个 token（如 'XZ4054H', 'GPT4'）
    - 单个中文字符 = 一个 token
    - 其他符号丢弃
    """
    tokens = []
    buf = []
    for ch in text.lower():
        if ch.isalnum() and ord(ch) < 128:
            buf.append(ch)
        else:
            if buf:
                tokens.append(''.join(buf)); buf = []
            if '\u4e00' <= ch <= '\u9fff':
                tokens.append(ch)
    if buf:
        tokens.append(''.join(buf))
    return tokens

# 演示
for s in ['Transformer 是 2017 年提出的', 'XZ4054H 锂电池规格书', 'RAG vs Agent', 'hello world！']:
    print(f'{s!r:40} -> {tokenize(s)}')

## 2. 手撸 BM25

**核心公式**（每个 query 词 q 对每个 doc D 的贡献）：

$$\text{score}(D, q) = \text{IDF}(q) \cdot \frac{\text{tf}(q, D) \cdot (k_1 + 1)}{\text{tf}(q, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgdl}})}$$

- `IDF(q) = log((N - df + 0.5) / (df + 0.5) + 1)` —— 罕见词权重高
- `tf` 用饱和函数处理 —— 一个词出现 10 次 vs 1 次的差别 < 10 vs 100
- `|D| / avgdl` —— 长 doc 应被适当惩罚
- 经验值 `k1 = 1.5, b = 0.75`

In [ ]:
class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1; self.b = b
        self.docs = docs
        self.doc_tokens = [tokenize(d) for d in docs]
        self.doc_len = [len(t) for t in self.doc_tokens]
        self.avg_dl = sum(self.doc_len) / max(1, len(self.doc_len))
        self.N = len(docs)
        # 倒排：每个词的 doc-frequency
        df = Counter()
        for tokens in self.doc_tokens:
            for t in set(tokens):
                df[t] += 1
        # 预存 IDF
        self.idf = {t: math.log((self.N - cnt + 0.5) / (cnt + 0.5) + 1.0) for t, cnt in df.items()}
        # 预存每个 doc 的词频
        self.doc_tf = [Counter(t) for t in self.doc_tokens]

    def score(self, query: str) -> np.ndarray:
        q_tokens = tokenize(query)
        scores = np.zeros(self.N, dtype=np.float32)
        for q in q_tokens:
            if q not in self.idf:
                continue
            idf_q = self.idf[q]
            for i in range(self.N):
                tf = self.doc_tf[i].get(q, 0)
                if tf == 0:
                    continue
                dl = self.doc_len[i]
                norm = tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * dl / max(1, self.avg_dl)))
                scores[i] += idf_q * norm
        return scores

    def search(self, query: str, top_k: int = 5) -> list[tuple[int, float]]:
        scores = self.score(query)
        order = np.argsort(-scores)[:top_k]
        return [(int(i), float(scores[i])) for i in order if scores[i] > 0]

In [ ]:
# Demo 语料：模拟一个电子元器件目录
DOCS = [
    '锂电池 XZ4054H 是一款高能量密度电池，容量 300mAh，循环寿命 500 次。',
    'XZ5352R 是一款 800mAh 锂电池，专为高放电场景设计。',
    'XZ4054H-NE1.11 是 XZ4054H 的升级版，集成保护电路，规格书 V1.11。',
    '电源管理芯片 PM8916 提供多路稳压输出，工作电压 3.3V。',
    'Transformer 是 2017 年由 Google 提出的注意力机制神经网络。',
    'RAG 是检索增强生成的简称，把外部知识接进 LLM 上下文。',
    '注意力机制（Attention）解决了 RNN 在长序列上的梯度衰减问题。',
    'PM8917 是 PM8916 的低功耗替代型号，待机电流降低 50%。',
]

bm25 = BM25(DOCS)
for q in ['XZ4054H 规格', '注意力机制', '低功耗电源']:
    print(f'\nBM25 query: {q!r}')
    for i, s in bm25.search(q, top_k=3):
        print(f'  [doc{i}] score={s:6.3f}  {DOCS[i]}')

## 3. dense 检索（embedding）

复用 16 号的套路。注意 OFFLINE 下 fake_embed 没语义，对「同义改写」无法召回。

In [ ]:
doc_vecs = np.vstack([embed(d) for d in DOCS])

def dense_search(query: str, top_k: int = 3) -> list[tuple[int, float]]:
    q = embed(query)
    sims = doc_vecs @ q
    order = np.argsort(-sims)[:top_k]
    return [(int(i), float(sims[i])) for i in order]

for q in ['XZ4054H 规格', '注意力机制', '低功耗电源']:
    print(f'\nDense query: {q!r}  (MODE={MODE})')
    for i, s in dense_search(q):
        print(f'  [doc{i}] sim={s:+.3f}  {DOCS[i]}')

## 4. RRF（Reciprocal Rank Fusion）—— 融合两路结果

**核心公式**（每个 doc 综合分数）：

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$$

- 取每个 doc 在所有排名里的「位置」（rank 从 1 开始）
- 用 `1/(k + rank)` 求倒数加权，k 一般 60
- **优势**：不需要标准化原始 score（BM25 score 与 cosine sim 量级完全不同）
- **唯一假设**：每路给出一个排名

In [ ]:
def rrf_fuse(*rankings: list[tuple[int, float]], k: int = 60, top_k: int = 5) -> list[tuple[int, float]]:
    """
    rankings: 多路检索结果，每路是 [(doc_id, score)] 列表（按 score 降序排）
    """
    rrf_scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    out = sorted(rrf_scores.items(), key=lambda x: -x[1])[:top_k]
    return [(d, s) for d, s in out]

def hybrid_search(query: str, top_k: int = 5, recall_k: int = 10) -> list[tuple[int, float]]:
    bm = bm25.search(query, top_k=recall_k)
    ds = dense_search(query, top_k=recall_k)
    return rrf_fuse(bm, ds, top_k=top_k)

for q in ['XZ4054H 规格', '注意力机制', '低功耗电源']:
    print(f'\nHybrid query: {q!r}')
    bm = bm25.search(q, 3)
    ds = dense_search(q, 3)
    hy = hybrid_search(q, 3)
    print(f'  BM25  top3 docs: {[d for d, _ in bm]}')
    print(f'  Dense top3 docs: {[d for d, _ in ds]}')
    print(f'  RRF   top3 docs: {[d for d, _ in hy]}')
    for i, s in hy:
        print(f'    [doc{i}] rrf={s:.4f}  {DOCS[i]}')

## 5. 各擅胜场 —— 量化对比

设计 10 题 eval set，每题标「期望的 doc_id」，看 BM25 / Dense / Hybrid 各自的 hit@3。

In [ ]:
EVAL = [
    # 精确关键词类（BM25 强）
    {'q': 'XZ4054H 规格书',           'expect': [0, 2]},
    {'q': 'XZ5352R 容量',             'expect': [1]},
    {'q': 'PM8916 电压',              'expect': [3]},
    {'q': 'PM8917 待机',              'expect': [7]},
    # 同义改写类（Dense 强）
    {'q': '高能量密度电池有哪些型号', 'expect': [0, 1, 2]},
    {'q': '检索增强生成的概念',        'expect': [5]},
    {'q': 'self-attention 是什么',     'expect': [4, 6]},
    # 混合（hybrid 兜底）
    {'q': '低功耗 PM 系列芯片',        'expect': [3, 7]},
    {'q': '锂电池 XZ4054H 升级版',     'expect': [0, 2]},
    {'q': 'transformer 与注意力',      'expect': [4, 6]},
]

def hit_at_k(retrieved: list[tuple[int, float]], expect: list[int], k: int = 3) -> bool:
    ids = [d for d, _ in retrieved[:k]]
    return any(e in ids for e in expect)

results = {'BM25': 0, 'Dense': 0, 'Hybrid (RRF)': 0}
details = []
for item in EVAL:
    bm = bm25.search(item['q'], top_k=10)
    ds = dense_search(item['q'], top_k=10)
    hy = rrf_fuse(bm, ds, top_k=10)
    row = {
        'q': item['q'],
        'BM25':         '✅' if hit_at_k(bm, item['expect']) else '❌',
        'Dense':        '✅' if hit_at_k(ds, item['expect']) else '❌',
        'Hybrid (RRF)': '✅' if hit_at_k(hy, item['expect']) else '❌',
    }
    details.append(row)
    for k in results:
        if row[k] == '✅':
            results[k] += 1

print(f'{"query":<35} {"BM25":>6} {"Dense":>6} {"Hybrid":>7}')
print('-' * 60)
for r in details:
    print(f'{r["q"][:35]:<35} {r["BM25"]:>6} {r["Dense"]:>6} {r["Hybrid (RRF)"]:>7}')
print('-' * 60)
n = len(EVAL)
print(f'{"hit@3":<35} {results["BM25"]/n*100:>5.0f}% {results["Dense"]/n*100:>5.0f}% {results["Hybrid (RRF)"]/n*100:>6.0f}%')
if MODE == 'OFFLINE':
    print('\n⚠ OFFLINE 下 Dense 用 fake_embed，「同义改写」全军覆没属正常。')
    print('  Hybrid 仍能受益于 BM25 兜底。切到 ONLINE 看真实差距。')

## 6. 工程实战要点

1. **召回放大 + 融合**：每路召 top-10 ~ 20，融合后取 top-5。**避免某路漏掉关键 doc**。
2. **k 取 60**：RRF 的 k 不敏感，60 是 IR 社区常用经验值。
3. **加 rerank 才是完整三段**：粗召（BM25 + dense）→ 融合（RRF）→ 精排（cross-encoder reranker）。见 notebook 22。
4. **生产实现**：Elasticsearch / OpenSearch 自带 BM25；Qdrant / Weaviate / Milvus 2.4+ 支持 hybrid；自建走 BM25 索引 + 向量库各 query 一遍再融合。
5. **何时 hybrid 不值得**：纯英文学术语料 / 全是自然语言无术语 → 纯 dense 可能也够。**先在 eval set 上对比**再决定。

## 深入思考

1. **BM25 比 TF-IDF 好在哪？**
   - TF 饱和（长 doc 里同一词出现 100 次不会被 100× 加分）+ 文档长度归一。是 TF-IDF 的「工程加强版」。
2. **RRF 为什么不直接加 score？**
   - BM25 score 量级 5–20，cosine 0–1，量级差 1-2 个数量级。直接加会让 BM25 把 dense 淹掉。RRF 只看排名，**与量级无关**，最稳健。
3. **`k1, b` 调参敏感吗？**
   - 不太敏感。`k1=1.5, b=0.75` 是经验默认，覆盖绝大多数场景。`b → 0` 完全不惩罚长 doc；`b → 1` 强惩罚。
4. **如果文档非常多（千万级），BM25 怎么加速？**
   - 用倒排索引（Lucene / Tantivy / Whoosh）。我们手撸的 `for q in q_tokens: for i in N` 是 O(Q × N)，倒排只算「query 命中的 doc」，跳到 O(Q × postings)。
5. **dense + dense（不同模型）也算 hybrid 吗？**
   - 算，叫 **mixture-of-encoders**。两个 embedding 模型擅长不同类型 → RRF 融合。但 dense + BM25 是「稠密 + 稀疏」最常见组合。

**改一改**：
- 把 RRF 的 `k` 改成 1，看排名靠后的 doc 是否被压得更厉害
- 加一种「同义改写 query」，看 BM25 是否会完全失效（应该会）

## 自检 ✅

- [ ] 默写 BM25 公式三个核心组件（IDF、TF 饱和、长度归一）。
- [ ] 解释「为什么 RRF 比直接加 score 更稳健」。
- [ ] 给一个 RAG 项目召回不准的 case，能立刻问「试过 hybrid 了吗」。
- [ ] 给一段中英混合文本，能现场写 5 行 tokenize 函数。
- [ ] 解释「先 recall 放大、后融合 / 精排」的两阶段范式。

## 下一步

进入 Stage 3 → [`../stage3_高级/22_rerank_hyde_multiquery.ipynb`](../stage3_高级/22_rerank_hyde_multiquery.ipynb)